# Tabela de Dominância Direta (#MSC e Vitória / Empate / Derrota)

Algoritmo principal: **ILS-VND**. Por instância usa-se a **média do objetivo nas 30 seeds** (maximização).

- **#MSC**: instâncias em que a média do ILS-VND alcança o melhor solução conhecida (MSC) (`best_solutions/best_objectives.csv`), com tolerância relativa `TOL`.
- **V / E / D**: instâncias em que o ILS-VND vence (V) / empata (E) / perde (D) contra cada outro algoritmo. Empate = diferença relativa ≤ `TOL`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks" / "utils" / "__init__.py").exists())
RESULTS = ROOT / "project" / "results"
BEST_CSV = ROOT / "best_solutions" / "best_objectives.csv"
IMG = ROOT / "notebooks" / "resultados" / "img"; IMG.mkdir(parents=True, exist_ok=True)
TEX = ROOT / "notebooks" / "resultados" / "tex"; TEX.mkdir(parents=True, exist_ok=True)

DATASETS = ["a", "b", "x"]
ALGOS = {"order_based": "OrderBased", "aisle_based": "AisleBased",
         "ils_vnd_all": "ILS-VND", "andre_feijo_all": "Santos e Baldotto (2025)"}
PRINCIPAL = "ILS-VND"
OPONENTES = [a for a in ALGOS.values() if a != PRINCIPAL]
TOL = 1e-4  # tolerância relativa (0,01%)

In [2]:
frames = []
for folder, label in ALGOS.items():
    for ds in DATASETS:
        for f in sorted((RESULTS / folder / ds).glob("instance_*.csv")):
            d = pd.read_csv(f, usecols=["objective"])
            d["algo"], d["dataset"], d["instance"] = label, ds, f.stem + ".txt"
            frames.append(d)
raw = pd.concat(frames, ignore_index=True)
best = pd.read_csv(BEST_CSV).rename(columns={"best_objective": "best"})
agg = (raw.groupby(["dataset", "instance", "algo"]).agg(media=("objective", "mean"), n=("objective", "size"))
          .reset_index().merge(best, on=["dataset", "instance"], how="left"))
print("Grupos com != 30 seeds:", int((agg.n != 30).sum()), "| sem best:", int(agg.best.isna().sum()))
M = agg.pivot(index=["dataset", "instance"], columns="algo", values="media")
B = best.set_index(["dataset", "instance"]).best.reindex(M.index)

Grupos com != 30 seeds: 0 | sem best: 0


In [3]:
def cmp(a, b):
    """+1 se a > b, 0 se empate (tol. relativa), -1 se a < b."""
    ref = np.maximum(np.abs(a), np.abs(b))
    return np.where(np.abs(a - b) <= TOL * ref, 0, np.sign(a - b)).astype(int)

bks = (M[PRINCIPAL] >= B * (1 - TOL))
res = pd.DataFrame(index=M.index)
res["msc"] = bks
for o in OPONENTES:
    res[o] = cmp(M[PRINCIPAL].values, M[o].values)

SUB = ["V", "E", "D"]  # Vitória, Empate, Derrota
linhas = {}
for ds in DATASETS + ["Total"]:
    r = res if ds == "Total" else res.loc[ds]
    lin = {("", "N"): len(r), ("", "#MSC"): int(r.msc.sum())}
    for o in OPONENTES:
        lin[(f"vs {o}", "V")] = int((r[o] > 0).sum())
        lin[(f"vs {o}", "E")] = int((r[o] == 0).sum())
        lin[(f"vs {o}", "D")] = int((r[o] < 0).sum())
    linhas["Total" if ds == "Total" else ds.upper()] = lin
tab = pd.DataFrame(linhas).T
tab.columns = pd.MultiIndex.from_tuples(tab.columns)
tab.index.name = "Dataset"
for o in OPONENTES:
    assert (tab[(f"vs {o}", "V")] + tab[(f"vs {o}", "E")] + tab[(f"vs {o}", "D")] == tab[("", "N")]).all()
display(tab.style.set_caption(f"Dominância direta do {PRINCIPAL} (30 seeds, tolerância relativa {TOL:.2%})")
        .set_table_styles([{"selector": "th", "props": [("text-align", "center"), ("border", "1px solid #999")]},
                           {"selector": "td", "props": [("text-align", "center")]}]))

## Exportação (PNG e LaTeX)

In [4]:
# --- PNG: cabeçalho de grupo (vs oponente) acima das subcolunas V / E / D ---
from matplotlib.patches import Rectangle
cols = list(tab.columns)
nc = len(cols)
sub = [s if g else n for (g, s), n in zip(cols, [c[1] for c in cols])]
fig, ax = plt.subplots(figsize=(0.75 * nc + 1.6, 0.4 * (len(tab) + 3) + 0.6)); ax.axis("off")
ax.set_title(f"Dominância direta do {PRINCIPAL} (V / E / D)", fontsize=10, pad=8)
tb = ax.table(cellText=[[str(v) for v in row] for row in tab.values], colLabels=sub,
              rowLabels=list(tab.index), cellLoc="center", loc="center")
tb.auto_set_font_size(False); tb.set_fontsize(8); tb.scale(1, 1.3)
for (r, c), cell in tb.get_celld().items():
    cell.set_edgecolor("#999")
    if r == 0 or c == -1:
        cell.set_facecolor("#eeeeee"); cell.set_text_props(weight="bold")
fig.canvas.draw()
inv = ax.transAxes.inverted()
def bbox(c):
    b = tb[0, c].get_window_extent(fig.canvas.get_renderer())
    (x0, y0), (x1, y1) = inv.transform((b.x0, b.y0)), inv.transform((b.x1, b.y1))
    return x0, y0, x1 - x0, y1 - y0
for o in OPONENTES:
    c0 = cols.index((f"vs {o}", "V"))
    x0, y0, w0, h = bbox(c0); x2, _, w2, _ = bbox(c0 + 2)
    ax.add_patch(Rectangle((x0, y0 + h), x2 + w2 - x0, h, transform=ax.transAxes, facecolor="#eeeeee", edgecolor="#999", lw=1.5, clip_on=False))
    ax.text(x0 + (x2 + w2 - x0) / 2, y0 + 1.5 * h, f"vs {o}", transform=ax.transAxes, ha="center", va="center", fontsize=8, fontweight="bold")
fig.savefig(IMG / "dominancia.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close(fig)

# --- LaTeX ---
def esc(s): return str(s).replace("#", r"\\#").replace("_", r"\\_")
nop = len(OPONENTES)
CW = r">{\centering\arraybackslash}p{0.9cm}"  # largura fixa p/ V, E e D (requer pacote array)
spec = "l | c | c | " + " | ".join([" ".join([CW] * 3)] * nop)
L = [r"\begin{table}[H]", r"  \centering", r"  \begin{minipage}{\textwidth}",
     rf"    \caption{{\label{{tab:dominancia}} Dominância direta do {PRINCIPAL} (V = vitórias, E = empates, D = derrotas) e \#MSC, médias de 30 sementes, tolerância relativa de {TOL:.2%}.}}",
     r"    \resizebox{\textwidth}{!}{", rf"      \begin{{tabular}}{{{spec}}}", r"        \hline",
     r"        \multirow{2}{*}{\textbf{Dataset}} & \multirow{2}{*}{\textbf{N}} & \multirow{2}{*}{\textbf{\#MSC}} & "
     + " & ".join(rf"\multicolumn{{3}}{{c{'|' if k < nop - 1 else ''}}}{{\textbf{{{esc(o)}}}}}" for k, o in enumerate(f"vs {x}" for x in OPONENTES)) + r" \\",
     "        " + " ".join(rf"\cline{{{4 + 3 * k}-{6 + 3 * k}}}" for k in range(nop)),
     r"        & & & " + " & ".join(r"\textbf{V} & \textbf{E} & \textbf{D}" for _ in OPONENTES) + r" \\",
     r"        \hline"]
for i, (idx, row) in enumerate(tab.iterrows()):
    pre = r"\rowcolor{gray!20} " if i % 2 else ""
    L.append(f"        {pre}{idx} & " + " & ".join((r"\phantom{0}" if v < 10 else "") + str(v) for v in row) + r" \\")
L += [r"        \hline", r"      \end{tabular}", r"    }", r"    \caption*{\footnotesize Fonte: Elaborado pelo autor, 2026.}",
      r"  \end{minipage}", r"\end{table}"]
(TEX / "tabela_dominancia.tex").write_text("\n".join(L) + "\n", encoding="utf-8")
print("\n".join(L))

\begin{table}[H]
  \centering
  \begin{minipage}{\textwidth}
    \caption{\label{tab:dominancia} Dominância direta do GVNS (V = vitórias, E = empates, D = derrotas) e \#MSC, médias de 30 sementes, tolerância relativa de 0.01%.}
    \resizebox{\textwidth}{!}{
      \begin{tabular}{l | c | c | >{\centering\arraybackslash}p{0.9cm} >{\centering\arraybackslash}p{0.9cm} >{\centering\arraybackslash}p{0.9cm} | >{\centering\arraybackslash}p{0.9cm} >{\centering\arraybackslash}p{0.9cm} >{\centering\arraybackslash}p{0.9cm} | >{\centering\arraybackslash}p{0.9cm} >{\centering\arraybackslash}p{0.9cm} >{\centering\arraybackslash}p{0.9cm}}
        \hline
        \multirow{2}{*}{\textbf{Dataset}} & \multirow{2}{*}{\textbf{N}} & \multirow{2}{*}{\textbf{\#MSC}} & \multicolumn{3}{c|}{\textbf{vs OrderBased}} & \multicolumn{3}{c|}{\textbf{vs AisleBased}} & \multicolumn{3}{c}{\textbf{vs Santos e Baldotto (2025)}} \\
        \cline{4-6} \cline{7-9} \cline{10-12}
        & & & \textbf{V} & \textbf{E} & \textbf{